# معمل تجارب الـ RAG متعدد الوسائط (Multimodal RAG Architecture)

أهلاً بك في أول معمل تجارب عملي لمرحلة الـ Multimodal RAG.

حتى الآن، كنا بنتعامل مع النصوص فقط (PDFs، وورد، وقواعد بيانات). لكن في الواقع، أغلب وثائق الشركات (زي التقارير المالية والتحليلات) بتحتوي على رسومات بيانية مهمة جداً فيها أرقام الإيرادات ومخططات معمارية.

في هذا المعمل، هنجرب بنفسنا خطوة بخطوة:
1. إزاي بنجهز ونمثل البيانات اللي فيها نصوص وصور معاً.
2. إزاي بنحول الرسم البياني لصيغة Base64 نقدر نبعتها للنموذج.
3. إزاي بنبني فضاء متجهات موحد يضم النصوص والصور ونبحث فيه باستخدام FAISS.
4. إزاي بنصيغ الرسالة المتكاملة اللي بتفهمها النماذج البصرية الكبيرة (Vision Models).

In [1]:
import io
import base64
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("تم تحميل جميع المكتبات الأساسية بنجاح.")

تم تحميل جميع المكتبات الأساسية بنجاح.


### 1. توليد رسم بياني تجريبي وتشفيره بـ Base64

تعال نتخيل إن عندنا تقرير مالي لشركة، وفيه رسم بياني بيوضح نمو الإيرادات السنوية من 2021 لحد 2024.
هنرسم المخطط باستخدام matplotlib ونحوله لسلسلة Base64.

In [2]:
fig, ax = plt.subplots(figsize=(6, 3))
years = ['2021', '2022', '2023', '2024']
revenue = [45.2, 58.7, 72.1, 95.4]
ax.bar(years, revenue, color='#2b5c8f')
ax.set_title('نمو الإيرادات السنوية بالمليون دولار')
ax.set_ylabel('الإيرادات ($M)')

buf = io.BytesIO()
plt.tight_layout()
plt.savefig(buf, format='png')
plt.close(fig)

chart_b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
print(f"تم إنشاء الرسم البياني وتشفيره بنجاح! طول كود الـ Base64: {len(chart_b64)} حرف.")

تم إنشاء الرسم البياني وتشفيره بنجاح! طول كود الـ Base64: 15308 حرف.


### 2. تجهيز الوثائق متعددة الوسائط (نصوص + صور)

دلوقتي هنبني قائمة بتمثل محتويات التقرير: فقرة نصية عن أداء الشركة، والرسم البياني للإيرادات، وفقرة نصية تانية عن البنية التحتية الهندسية.

In [3]:
documents = [
    {
        "id": "doc_text_1",
        "type": "text",
        "text": "حققت الشركة نمواً قياسياً في الأرباح التشغيلية لعام 2024 بفضل التوسع في الحلول السحابية.",
        "source": "annual_report.pdf",
        "image_b64": None
    },
    {
        "id": "doc_image_1",
        "type": "image",
        "text": "رسم بياني يوضح إيرادات شركة أكمي السنوية من 2021 إلى 2024 وارتفاعها من 45.2 إلى 95.4 مليون دولار.",
        "source": "annual_report.pdf",
        "image_b64": chart_b64
    },
    {
        "id": "doc_text_2",
        "type": "text",
        "text": "أكمل فريق الهندسة ترقية خوادم قواعد البيانات وتقليص زمن الاستجابة بنسبة 60%.",
        "source": "annual_report.pdf",
        "image_b64": None
    }
]

print(f"تم تجهيز {len(documents)} عناصر (فقرات نصية ورسوم بيانية).")

تم تجهيز 3 عناصر (فقرات نصية ورسوم بيانية).


### 3. تحميل نموذج التضمين وبناء فضاء المتجهات

هنستخدم نموذج محلي من SentenceTransformers لتضمين النصوص وأوصاف الرسوم البيانية في متجهات متطابقة الأبعاد.

In [4]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
dim = embedder.get_sentence_embedding_dimension()
print(f"نموذج التضمين جاهز، أبعاد المتجه الواحد: {dim}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

نموذج التضمين جاهز، أبعاد المتجه الواحد: 384


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_13676\301363404.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = embedder.get_sentence_embedding_dimension()


### 4. فهرسة الوثائق في قاعدة بيانات FAISS

تعال نحول النصوص والأشكال إلى متجهات طبيعية (L2 Normalized) ونضيفها في فهرس FAISS FlatIP (الذي يطابق الـ Cosine Similarity).

In [5]:
texts_to_embed = [doc["text"] for doc in documents]
vectors = embedder.encode(texts_to_embed, convert_to_numpy=True, normalize_embeddings=True)

index = faiss.IndexFlatIP(dim)
index.add(vectors.astype(np.float32))

print(f"تمت فهرسة {index.ntotal} عناصر داخل قاعدة بيانات FAISS بنجاح.")

تمت فهرسة 3 عناصر داخل قاعدة بيانات FAISS بنجاح.


### 5. تجربة الاسترجاع العابر للوسائط (Cross-Modal Search)

طب لو سألنا سؤال باللغة العربية: *"عايز تفاصيل ومخطط الإيرادات السنوية للشركة؟"*
تعال نشوف إزاي الفهرس هيرجع لنا الفقرة النصية ومعاها الرسم البياني في نفس الوقت!

In [6]:
query = "عايز تفاصيل ومخطط الإيرادات السنوية للشركة"
query_vec = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)

scores, indices = index.search(query_vec, k=2)

print("نتائج الاسترجاع:")
print("=" * 50)
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
    matched_doc = documents[idx]
    print(f"النتيجة {rank} (درجة التشابه: {score:.4f}):")
    print(f"- النوع: {matched_doc['type']}")
    print(f"- المحتوى: {matched_doc['text']}")
    if matched_doc['type'] == 'image':
        print(f"- تم إرفاق صورة الرسم البياني (Base64 متوفرة: {len(matched_doc['image_b64'])} حرف)")
    print("-" * 50)

نتائج الاسترجاع:
النتيجة 1 (درجة التشابه: 0.6410):
- النوع: text
- المحتوى: حققت الشركة نمواً قياسياً في الأرباح التشغيلية لعام 2024 بفضل التوسع في الحلول السحابية.
--------------------------------------------------
النتيجة 2 (درجة التشابه: 0.5987):
- النوع: image
- المحتوى: رسم بياني يوضح إيرادات شركة أكمي السنوية من 2021 إلى 2024 وارتفاعها من 45.2 إلى 95.4 مليون دولار.
- تم إرفاق صورة الرسم البياني (Base64 متوفرة: 15308 حرف)
--------------------------------------------------


### 6. تركيب رسالة الـ Multimodal LLM النهائية

النماذج البصرية (زي GPT-4o أو Gemini) مش بتستقبل برومبت نصي عادي بس، بل بتاخد قائمة من كتل النصوص وكتل الصور المرمزة بـ Base64.
تعال نشوف شكل الرسالة المجمعة اللي بتوصل للنموذج.

In [7]:
message_blocks = [
    {
        "type": "text",
        "text": f"أجب عن سؤال المستخدم بناءً على السياق والصور المرفقة أدناه:\nالسؤال: {query}\n\nالسياق المسترجع:"
    }
]

for idx in indices[0]:
    item = documents[idx]
    if item["type"] == "text":
        message_blocks.append({"type": "text", "text": f"[نص]: {item['text']}"})
    elif item["type"] == "image":
        message_blocks.append({"type": "text", "text": f"[صورة توضيحية]: {item['text']}"})
        message_blocks.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{item['image_b64'][:40]}... (تم اقتطاع باقي الكود للعرض)"}
        })

print("هيكل رسالة الـ Multimodal Prompt:")
for i, block in enumerate(message_blocks):
    print(f"كتلة {i+1} ({block['type']}): {list(block.keys())}")

هيكل رسالة الـ Multimodal Prompt:
كتلة 1 (text): ['type', 'text']
كتلة 2 (text): ['type', 'text']
كتلة 3 (text): ['type', 'text']
كتلة 4 (image_url): ['type', 'image_url']


### 7. جرب بنفسك!

دلوقتي دورك للتجربة الحرة:
1. جرب تسأل سؤال مختلف، مثلاً عن البنية التحتية أو السيرفرات: `"ما هي تحديثات قواعد البيانات؟"`.
2. لاحظ هل النتيجة الأولى هتكون النص أم الرسم البياني؟
3. جرب تضيف عنصر رابع في قائمة `documents` بصورة جديدة أو نص إضافي وشوف النتيجة بنفسك.

In [8]:
# جرب سؤالك هنا
my_query = "ما هي تحديثات قواعد البيانات والبنية التحتية؟"

my_vec = embedder.encode([my_query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
my_scores, my_indices = index.search(my_vec, k=1)

best_match = documents[my_indices[0][0]]
print(f"السؤال: {my_query}")
print(f"أقرب مستند مسترجع: {best_match['text']}")
print(f"نوعه: {best_match['type']}")
print(f"درجة التشابه: {my_scores[0][0]:.4f}")

السؤال: ما هي تحديثات قواعد البيانات والبنية التحتية؟
أقرب مستند مسترجع: حققت الشركة نمواً قياسياً في الأرباح التشغيلية لعام 2024 بفضل التوسع في الحلول السحابية.
نوعه: text
درجة التشابه: 0.6583
